# Phase 2 — Step 3: A2 (decoder + copy) + **full fine-tuning** for Legal NER (experiment 1, KD teacher)

**What happens here**

1. Load the Phase-1 A2 model's weights (`model_final.pt`) into a _backbone_: token ids → one 640-number vector per token.
   The LM head and the copy mechanism are dropped: they predict the next token, not a tag.
2. Add a new **head**: `Linear(640 → 25)`, i.e. 25 scores per token, one per BIO tag.
3. **Full fine-tuning** = every weight (backbone + head) is updated by training on NER labels.
4. Attention is switched from **causal** (left context only, how A2 was pretrained) to **bidirectional**
   (both sides), because the label of a name often depends on words _after_ it.

**Two stages**

- **Stage A**: train on fold 1, validate on fold 2 after every epoch, keep the best epoch (early stopping).
  These fold-2 numbers are what we report and compare between experiments.
- **Stage B**: retrain from Phase-1 weights on **fold 1 + fold 2** (twice the data) for the best number of epochs from Stage A,
  then predict the test set → `submission.zip`. (No validation possible here: all labelled data is used for training.)

Every stage is cached: re-running the notebook loads finished models / predictions instead of redoing them.
Training checkpoints are saved after every epoch and resumed automatically.

**Ablation switches** (cell 1): `CAUSAL`, `NORMALIZE`, `HEADER`. Each setting gets its own run folder.
For an ablation, set `FINAL = False` (Stage A only, no submission).


## 0. Settings


In [8]:
import sys, json, time, importlib
from pathlib import Path
import numpy as np
import torch
from tokenizers import Tokenizer

# ---- experiment switches ------------------------------------------------------------
PHASE1_RUN = "A2_copy1_mtp0_d640_L10_sc70tr30_200M"   # decoder + copy (best Phase-1 model)
METHOD     = "full"      # full fine-tuning: all weights trainable
CAUSAL     = True       # False = bidirectional attention (our choice); True = keep Phase-1 causal attention (ablation)
NORMALIZE  = True        # offset-preserving cleaning (Step 2); False = raw text (ablation)
HEADER     = 0           # 0 = plain windows; 256 = "header memory" windows (our own idea, ablation)
FINAL      = False        # True = also Stage B (train on fold1+fold2) + test submission

CFG = dict(
    max_len=1536, stride=1024,          # windows (Phase-1 context length)
    epochs=20, patience=4,              # Stage A: at most 20 epochs, stop after 4 epochs without improvement
    micro_batch=2, grad_accum=2,        # 4 windows (6,144 tokens) per optimizer step
    lr=1e-4, head_lr=1e-3,              # backbone / new head learning rates
    weight_decay=0.01, warmup_frac=0.1, grad_clip=1.0,
    dropout=0.1, class_weights=None, seed=42,
)

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import ner_utils as nu, ner_models as nm
importlib.reload(nu); importlib.reload(nm)

RUN_NAME = f"A2_{METHOD}_{'causal' if CAUSAL else 'bidir'}_{'norm' if NORMALIZE else 'raw'}_hm{HEADER}"
RUN_DIR = PROJECT_ROOT / "checkpoints" / "phase2" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
json.dump({**CFG, "phase1_run": PHASE1_RUN, "method": METHOD, "causal": CAUSAL,
           "normalize": NORMALIZE, "header": HEADER}, open(RUN_DIR / "config.json", "w"), indent=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
print("run:", RUN_NAME, "| device:", device, "| precision:", AMP)

run: A2_full_causal_norm_hm0 | device: cuda | precision: torch.bfloat16


## 1. Data → tokens → BIO tags → windows

Exactly the Step-2 pipeline. Training windows are 1536 tokens long; fold 1 gives ~223 of them.


In [9]:
data = nu.load_ner_data(PROJECT_ROOT)
LABELS = data["labels"]
TAGS, TAG2ID = nu.make_tags(LABELS)
tokenizer = Tokenizer.from_file(str(PROJECT_ROOT / "data" / "tokenizer" / "bpe32k.json"))

docs = {f: nu.encode_docs(data["texts"][f], tokenizer, normalize=NORMALIZE, desc=f) for f in ("fold1", "fold2")}
test_docs = nu.encode_docs(data["test"], tokenizer, normalize=NORMALIZE, desc="test")
tags = {f: [nu.spans_to_tags(d, data["golds"][f][d.doc_id], TAG2ID) for d in docs[f]] for f in docs}

X1, Y1 = nu.make_training_windows(docs["fold1"], tags["fold1"], CFG["max_len"], CFG["stride"], HEADER)
X12, Y12 = nu.make_training_windows(docs["fold1"] + docs["fold2"], tags["fold1"] + tags["fold2"],
                                    CFG["max_len"], CFG["stride"], HEADER)
print(f"training windows: fold1 {len(X1)} | fold1+fold2 {len(X12)} | window length {len(X1[0])}")
print(f"tokens labelled as entity (not O) in fold1: {np.mean(np.concatenate(Y1) > 0):.1%}")

training windows: fold1 223 | fold1+fold2 446 | window length 1536
tokens labelled as entity (not O) in fold1: 8.8%


## 2. Model: Phase-1 backbone + new tagging head

`load_phase1_backbone` copies the Phase-1 weights with `strict=True` (fails loudly if any backbone weight is missing).
It reports what was dropped (LM head, copy mechanism). Then a quick check that one window runs.


In [10]:
def build_model():
    backbone, p1cfg, info = nm.load_phase1_backbone(PROJECT_ROOT / "checkpoints" / "phase1" / PHASE1_RUN)
    model = nm.NERTagger(backbone, len(TAGS), dropout=CFG["dropout"], causal=CAUSAL)
    return model.to(device), info

model, info = build_model()
total, trainable = nm.count_params(model)
print("loaded:", info)
print(f"parameters: {total / 1e6:.2f}M total, {trainable / 1e6:.2f}M trainable "
      f"(head: {sum(p.numel() for p in model.head.parameters()):,})")
with torch.no_grad(), torch.autocast(device.type, dtype=AMP, enabled=AMP != torch.float32):
    out = model(torch.from_numpy(X1[0][None]).long().to(device))
print("one window ->", tuple(out.shape), "(batch, tokens, tags)")

loaded: {'arch': 'A2', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 5, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'lm_head']}
parameters: 70.07M total, 70.07M trainable (head: 16,025)
one window -> (1, 1536, 25) (batch, tokens, tags)


## 3. Stage A — train on fold 1, validate on fold 2 after every epoch

`evaluate` runs the full prediction pipeline on fold 2 (windows → model → merge → Viterbi → trim → offsets)
and scores it with the strict scorer. The epoch with the best validation macro-F1 is kept.


In [11]:
def evaluate(m=None, docs_=None, golds_=None):
    m = m or model
    docs_ = docs_ or docs["fold2"]
    golds_ = golds_ or data["golds"]["fold2"]
    scorer = nm.make_scorer(m, device, AMP)
    pred = nu.predict_spans(docs_, scorer, TAGS, CFG["max_len"], CFG["stride"], HEADER)
    m.train()
    return nu.strict_scores(golds_, pred, LABELS)

t0 = time.time()
stateA = nm.train_tagger(model, X1, Y1, RUN_DIR, CFG, device, tag="fold1", eval_fn=evaluate, amp_dtype=AMP)
print(f"best epoch {stateA['best_epoch']} | best val macro-F1 {stateA['best']:.4f} | cell time {time.time() - t0:.0f}s")

epoch  1 | train loss 1.1723 | 12s | val macro-F1 0.2829 micro-F1 0.4489  <- best


epoch  2 | train loss 0.0880 | 12s | val macro-F1 0.3882 micro-F1 0.5154  <- best


epoch  3 | train loss 0.0416 | 12s | val macro-F1 0.4636 micro-F1 0.5703  <- best


epoch  4 | train loss 0.0238 | 12s | val macro-F1 0.4975 micro-F1 0.5934  <- best


epoch  5 | train loss 0.0076 | 12s | val macro-F1 0.4752 micro-F1 0.5856


epoch  6 | train loss 0.0028 | 12s | val macro-F1 0.4785 micro-F1 0.5900


epoch  7 | train loss 0.0030 | 12s | val macro-F1 0.4737 micro-F1 0.6003


epoch  8 | train loss 0.0020 | 12s | val macro-F1 0.4784 micro-F1 0.5929
early stop: no improvement for 4 epochs
training 'fold1' finished -> loaded best epoch 4
best epoch 4 | best val macro-F1 0.4975 | cell time 147s


## 4. Stage A results on fold 2 (the numbers for the report)

Loads the best fold-1 model, predicts fold 2 once (cached in `val_pred.json`), and shows:

- strict scores per label;
- the effect of our decoding choices on the **same** model outputs: argmax vs Viterbi, trim rule on/off;
- the error breakdown (which kind of mistakes remain).


In [12]:
VAL_PRED, VAL_LP = RUN_DIR / "val_pred.json", RUN_DIR / "val_logprobs.npz"
if VAL_PRED.exists() and VAL_LP.exists():
    val_pred = {k: [tuple(x) for x in v] for k, v in json.load(open(VAL_PRED)).items()}
    val_lp = dict(np.load(VAL_LP))
    print("[cache] loaded fold-2 predictions")
else:
    model.load_state_dict(torch.load(RUN_DIR / "best_fold1.pt", map_location=device))
    lp = nu.doc_log_probs(docs["fold2"], nm.make_scorer(model, device, AMP), CFG["max_len"], CFG["stride"], HEADER)
    val_lp = {k: v for k, v in lp.items() if v is not None}
    val_pred = nu.decode_docs(docs["fold2"], lp, TAGS)
    np.savez_compressed(VAL_LP, **val_lp)
    json.dump(val_pred, open(VAL_PRED, "w"))
gold2 = data["golds"]["fold2"]
val_scores = nu.strict_scores(gold2, val_pred, LABELS)
nu.print_scores(val_scores, f"{RUN_NAME} — fold 2 (validation):")

lp_all = {d.doc_id: val_lp.get(d.doc_id) for d in docs["fold2"]}
print("\ndecoding choices on the same model outputs (fold-2 macro-F1):")
decoding = {}
for dec in ("argmax", "viterbi"):
    for trim in (False, True):
        sc = nu.strict_scores(gold2, nu.decode_docs(docs["fold2"], lp_all, TAGS, decode=dec, trim=trim), LABELS)
        decoding[f"{dec}_trim{int(trim)}"] = sc["macro_f1"]
        print(f"  {dec:8s} trim={str(trim):5s} -> {sc['macro_f1']:.4f}")

cats, confusion, examples = nu.error_breakdown(gold2, val_pred, data["texts"]["fold2"], n_examples=3)
print("\nerror breakdown:", dict(cats.most_common()))
print("most confused labels (gold -> predicted):", confusion.most_common(8))
for cat, ex in examples.items():
    for doc_id, s, e, lab, ctx in ex:
        print(f"  [{cat}] {lab}: ...{ctx}...")
json.dump({"val": {k: v for k, v in val_scores.items() if k != "per_label"},
           "per_label": {l: round(v["f1"], 4) for l, v in val_scores["per_label"].items()},
           "decoding": decoding, "errors": dict(cats), "best_epoch": stateA["best_epoch"],
           "history": stateA["history"]}, open(RUN_DIR / "val_scores.json", "w"), indent=1)

A2_full_causal_norm_hm0 — fold 2 (validation):
  macro-F1 0.4975 | macro-P 0.5176 | macro-R 0.4992 | F1(of macro P,R) 0.5082 | micro-F1 0.5934
  label           P      R     F1    TP    FP    FN  gold
  APP         0.218  0.290  0.249    58   208   142   200
  RESP        0.316  0.163  0.215    31    67   159   190
  A.COUNSEL   0.485  0.504  0.494    63    67    62   125
  R.COUNSEL   0.500  0.505  0.503    49    49    48    97
  JUDGE       0.612  0.697  0.651    85    54    37   122
  WIT         0.432  0.158  0.232    19    25   101   120
  AUTH        0.503  0.316  0.388   148   146   320   468
  COURT       0.780  0.789  0.784   440   124   118   558
  STAT        0.664  0.690  0.676   507   257   228   735
  PREC        0.368  0.515  0.429   288   495   271   559
  DATE        0.946  0.936  0.941   713    41    49   762
  CASENO      0.388  0.427  0.407   144   227   193   337

decoding choices on the same model outputs (fold-2 macro-F1):
  argmax   trim=False -> 0.3577
  argmax

## 5. Stage B — retrain on fold 1 + fold 2, predict the test set, write the submission

Starts again from the Phase-1 weights (fresh model) and trains on all 70 documents for the **same number of epochs**
(passes over the data) as the best Stage-A epoch. Skipped when `FINAL = False`.


In [13]:
SUB_DIR = RUN_DIR / "submission"
if not FINAL:
    print("FINAL = False -> ablation run, no Stage B / submission")
elif (SUB_DIR / "submission.zip").exists():
    nu.check_submission(SUB_DIR / "submission.zip", data["test"], LABELS)
    print("[cache] submission already written")
else:
    model, _ = build_model()                                     # fresh Phase-1 weights
    cfgB = {**CFG, "epochs": stateA["best_epoch"]}
    stateB = nm.train_tagger(model, X12, Y12, RUN_DIR, cfgB, device, tag="full", eval_fn=None, amp_dtype=AMP)
    test_pred = nu.predict_spans(test_docs, nm.make_scorer(model, device, AMP), TAGS, CFG["max_len"], CFG["stride"], HEADER)
    zp = nu.write_submission(test_pred, data["test"], LABELS, SUB_DIR)
    counts = {}
    for spans in test_pred.values():
        for s, e, l in spans:
            counts[l] = counts.get(l, 0) + 1
    print("test entities per label:", dict(sorted(counts.items(), key=lambda kv: -kv[1])))

FINAL = False -> ablation run, no Stage B / submission


## 6. Log the experiment (one line per run in `experiments/phase2/experiments.jsonl`)


In [14]:
if RUN_NAME in {r.get("run") for r in nu.read_experiments(PROJECT_ROOT)}:
    print("[cache] already logged")
else:
    total, trainable = nm.count_params(model)
    rec = {"run": RUN_NAME, "phase1": PHASE1_RUN, "method": METHOD, "causal": CAUSAL, "normalize": NORMALIZE,
           "header": HEADER, "params_total": total, "params_trainable": trainable,
           "best_epoch": stateA["best_epoch"], "val_macro_f1": round(val_scores["macro_f1"], 4),
           "val_micro_f1": round(val_scores["micro_f1"], 4),
           "val_per_label": {l: round(v["f1"], 4) for l, v in val_scores["per_label"].items()},
           "train_sec_fold1": sum(h["train_sec"] for h in stateA["history"]),
           "submission": str((RUN_DIR / "submission" / "submission.zip").relative_to(PROJECT_ROOT)) if FINAL else None,
           "time": time.strftime("%Y-%m-%d %H:%M")}
    nu.log_experiment(PROJECT_ROOT, rec)
    print("logged:", rec)

logged: {'run': 'A2_full_causal_norm_hm0', 'phase1': 'A2_copy1_mtp0_d640_L10_sc70tr30_200M', 'method': 'full', 'causal': True, 'normalize': True, 'header': 0, 'params_total': 70071065, 'params_trainable': 70071065, 'best_epoch': 4, 'val_macro_f1': 0.4975, 'val_micro_f1': 0.5934, 'val_per_label': {'APP': 0.2489, 'RESP': 0.2153, 'A.COUNSEL': 0.4941, 'R.COUNSEL': 0.5026, 'JUDGE': 0.6513, 'WIT': 0.2317, 'AUTH': 0.3885, 'COURT': 0.7843, 'STAT': 0.6765, 'PREC': 0.4292, 'DATE': 0.9406, 'CASENO': 0.4068}, 'train_sec_fold1': 97.2, 'submission': None, 'time': '2026-10-09 16:49'}


## What to paste back

Outputs of cells 2–6 (the epoch lines from training are enough; no need for the progress bars).
If you see `CUDA out of memory`, set `micro_batch=1, grad_accum=4` (same 4 windows per step) and re-run.
